# Week 1 · Python II: classes, errors, imports

Why this matters for the exam: every PyTorch model is a **class**. By the end of this notebook you'll be able to read `class Model(nn.Module): ...` line by line. Days: sections 1–4 on Monday, sections 5–8 on Tuesday.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **class**: A blueprint for making objects. `list` is a class, and so is `dict`.
- **object / instance**: One thing made from a class. `[1, 2]` is one instance of the class `list`.
- **attribute**: A variable stored *inside* an object, written `obj.name`, e.g. `c.count`.
- **method**: A function defined inside a class, called with a dot: `c.add(5)`. You've been using these since Week 0 (`.append`, `.split`).
- **`self`**: Inside a method, `self` means *the object this method was called on*.
- **`__init__`**: A special method that runs **automatically when you create** an object. It sets up the attributes.
- **inheritance**: Making a new class based on an existing one. The new **child** class gets everything from the **parent**, and can change or add things.
- **`super()`**: Inside a child class, `super()` refers to the parent class. You use it to run the parent's version of a method.
- **exception**: Python's word for an error that happens while the code runs (e.g. `ValueError`).

## 1. Objects: you've been using them all along

A **class** bundles **data** together with **the functions that work on that data**. A list is an object: its data is the items, and its methods are `.append`, `.pop`, and so on.

In [1]:
nums = [3, 1, 2]
print(type(nums))      # → <class 'list'>   nums is an object of the class list
nums.append(5)         # a method: a function that belongs to the list and changes ITS data
print(nums)            # → [3, 1, 2, 5]

<class 'list'>
[3, 1, 2, 5]


Now you'll write your own classes, which is exactly what PyTorch asks you to do for every model.

## 2. Your first class

In [19]:
class Counter:
    def __init__(self):          # runs automatically when a Counter is created
        self.count = 0           # an attribute: data stored inside this object

    def add(self, n=1):          # a method
        self.count = self.count + n

c = Counter()                    # create an object; Python runs __init__ here
c.add()                          # count: 0 → 1
c.add(5)                         # count: 1 → 6
print(c.count)                   # → 6

6


Read it line by line:

| Line | What it means |
|---|---|
| `class Counter:` | start a blueprint called Counter (class names start with a capital letter) |
| `def __init__(self):` | the setup method; two underscores on each side |
| `self.count = 0` | store an attribute called `count` inside *this* object |
| `def add(self, n=1):` | a method. Its first input is always `self`. |
| `c = Counter()` | make one object. The `()` calls the class, which runs `__init__`. |
| `c.add(5)` | call the method on `c`. Inside `add`, `self` is `c` and `n` is 5. |

**What `self` really is.** When you write `c.add(5)`, Python secretly calls `Counter.add(c, 5)`. The object before the dot becomes `self`. That's why every method lists `self` first, even though you never pass it yourself.

In [3]:
c = Counter()
Counter.add(c, 3)       # exactly what c.add(3) does behind the scenes
print(c.count)          # → 3

3


**Each object has its own data.** Two counters don't share a count:

In [4]:
c1 = Counter()
c2 = Counter()
c1.add(10)
print(c1.count, c2.count)    # → 10 0

10 0


> ⚠️ **Common mistake.** Writing `count = 0` instead of `self.count = 0` inside `__init__`. Without `self.` it's just a temporary variable that disappears when `__init__` ends, so `c.count` later gives `AttributeError`.

**Quick check:** In `c.add(5)`, what are `self` and `n` inside `add`?

<details><summary>Answer</summary>

`self` is `c` (the object before the dot), and `n` is `5`.

</details>

## 3. `__init__` with inputs, and methods that return values

Usually you pass starting values when you create the object. They go to `__init__` (after `self`):

In [5]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner           # store the inputs as attributes
        self.balance = balance

    def deposit(self, amount):
        self.balance = self.balance + amount
        return self.balance          # a method can return a value, like any function

    def summary(self):
        return f"{self.owner}: {self.balance} won"

acct = BankAccount("Kim", 1000)      # owner="Kim", balance=1000 → passed to __init__
print(acct.deposit(500))             # → 1500
print(acct.summary())                # → Kim: 1500 won
print(acct.balance)                  # → 1500     attributes can be read directly

1500
Kim: 1500 won
1500


**Your turn.** Write a class `Rectangle` that takes `width` and `height` in `__init__`, and has a method `area()` that returns width × height. `Rectangle(3, 4).area()` should be `12`.

<details><summary>Show one answer</summary>

`def __init__(self, width, height): self.width = width; self.height = height` and `def area(self): return self.width * self.height`

</details>

In [21]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

print(Rectangle(3, 4).area())   # remove the # once your class is written: should print 12

12


## 4. Special methods: `__repr__`, `__len__`, `__getitem__`

Methods with double underscores on both sides (people say "dunder" methods) let your object work with Python's built-in tools. Printing a plain object isn't helpful:

In [7]:
print(acct)    # → something like <__main__.BankAccount object at 0x10a3...>

`__repr__` controls what `print` shows:

In [8]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"Point({self.x}, {self.y})"

p = Point(3, 4)
print(p)             # → Point(3, 4)

Point(3, 4)


`__len__` makes `len(obj)` work, and `__getitem__` makes `obj[i]` work:

In [9]:
class Shelf:
    def __init__(self, books):
        self.books = books
    def __len__(self):               # called by len(shelf)
        return len(self.books)
    def __getitem__(self, i):        # called by shelf[i]
        return self.books[i]

shelf = Shelf(["ISL", "d2l", "UDL"])
print(len(shelf))    # → 3
print(shelf[0])      # → ISL

3
ISL


**This is exactly what a PyTorch `Dataset` is**: a class with `__len__` (how many samples) and `__getitem__` (give me sample `i`). In Week 5 you'll write one.

**Quick check:** Which method does `len(obj)` call, and which does `obj[2]` call?

<details><summary>Answer</summary>

`obj.__len__()` and `obj.__getitem__(2)`.

</details>

## 5. Inheritance: a class built on another class

A **child** class lists its **parent** in parentheses. It automatically gets all of the parent's methods. It can **override** (replace) some of them, and add new ones.

In [10]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return "..."
    def intro(self):
        return f"{self.name} says {self.speak()}"

class Dog(Animal):             # Dog is a child of Animal
    def speak(self):           # override: Dog's own version replaces Animal's
        return "woof"

d = Dog("Rex")                 # Dog has no __init__, so it uses Animal's
print(d.intro())               # → Rex says woof

Rex says woof


Notice that `intro` comes from `Animal`, but when it calls `self.speak()`, `self` is a Dog, so Dog's `speak` runs. Python always looks in the object's **own** class first, then in the parent.

**`super()`**: when the child needs its own `__init__`, it must still run the parent's `__init__` so the parent's attributes get set up:

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

class Student(Person):
    def __init__(self, name, school):
        super().__init__(name)        # run Person's __init__ → sets self.name / Notice you don’t pass self to super().__init__(name). super() already knows which object it’s working on.
        self.school = school          # then add the child's own attribute

s = Student("Kim", "Yonsei")
print(s.name, s.school)               # → Kim Yonsei

Kim Yonsei


If you forget the `super().__init__(...)` line, the parent's setup never runs:

In [12]:
class BadStudent(Person):
    def __init__(self, name, school):
        self.school = school          # forgot super().__init__(name)

b = BadStudent("Lee", "Yonsei")
try:
    print(b.name)
except AttributeError as e:
    print("Error:", e)                # → Error: 'BadStudent' object has no attribute 'name'

Error: 'BadStudent' object has no attribute 'name'


> ⚠️ **Common mistake.** Forgetting `super().__init__()` in a child class's `__init__`. In PyTorch this gives a confusing error the moment you add a layer, so it's the first thing to check.

**Quick check:** `class Cat(Animal): def speak(self): return "meow"`. What is `Cat("Nabi").intro()`?

<details><summary>Answer</summary>

`'Nabi says meow'`

</details>

**Your turn.** Write `Square(Rectangle)` whose `__init__` takes only `side` and calls `super().__init__(side, side)`. Then `Square(5).area()` should be `25` without writing `area` again. (Use your `Rectangle` from section 3, so run that cell first.)

<details><summary>Show one answer</summary>

`class Square(Rectangle):` / `def __init__(self, side): super().__init__(side, side)`

</details>

In [28]:
class Square(Rectangle): 
    def __init__(self,side):
        self.side = super().__init__(side,side)


s = Square(5)
print(s.area())

25


## 6. Reading a PyTorch model (preview of Week 5)

Everything in this cell is something you've now learned. Read it before running it:

In [14]:
import torch
from torch import nn

class TinyModel(nn.Module):              # inherit from PyTorch's Module class
    def __init__(self):
        super().__init__()               # run nn.Module's setup (required!)
        self.layer = nn.Linear(3, 1)     # an attribute holding a layer: 3 inputs → 1 output

    def forward(self, x):                # a method: what the model does with input x
        return self.layer(x)

model = TinyModel()                      # __init__ runs
x = torch.ones(2, 3)                     # 2 samples, 3 numbers each
print(model(x).shape)                    # → torch.Size([2, 1])   one output per sample

torch.Size([2, 1])


| Line | Which idea from this notebook |
|---|---|
| `class TinyModel(nn.Module):` | inheritance (section 5) |
| `super().__init__()` | running the parent's setup (section 5) |
| `self.layer = ...` | an attribute (section 2) |
| `def forward(self, x):` | a method (section 2) |
| `model(x)` | calling the object runs `forward`. nn.Module defines a special method for this, the same idea as `__len__` in section 4. |

## 7. Errors on purpose: `try` / `except` and `raise`

**Reading an error message:** look at the **last line** first. It names the kind of error and says what went wrong. The lines above it show where.

**`try` / `except`** runs some code, and if a particular error happens, runs a backup plan instead of crashing:

In [15]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:          # only catches this kind of error
        return None

print(safe_divide(6, 3))     # → 2.0
print(safe_divide(6, 0))     # → None

2.0
None


**`raise`** stops your function on purpose with a clear message when the input makes no sense. This is better than silently returning a wrong answer:

In [16]:
def withdraw(balance, amount):
    if amount > balance:
        raise ValueError(f"cannot withdraw {amount}, balance is only {balance}")
    return balance - amount

print(withdraw(100, 30))     # → 70
try:
    withdraw(100, 500)
except ValueError as e:
    print("Error:", e)       # → Error: cannot withdraw 500, balance is only 100

70
Error: cannot withdraw 500, balance is only 100


| Error | Usually means |
|---|---|
| `NameError` | a name that doesn't exist yet: a typo, or a cell you didn't run |
| `TypeError` | the wrong kind of value, e.g. `"3" + 4`, or calling something that isn't a function |
| `IndexError` | a position past the end of a list |
| `KeyError` | a dict key that isn't there |
| `AttributeError` | `obj.something` doesn't exist (a typo, or a missing `self.` / `super().__init__()`) |
| `ValueError` | the right type but a bad value, e.g. `int("abc")` |

## 8. Imports, and functions with any number of inputs

**Imports** bring in code from other modules (files). Three forms you'll see everywhere:

In [17]:
import math                       # use as math.sqrt
from collections import Counter   # bring in one name directly
import numpy as np                # import with a short nickname (always np for NumPy)

print(math.sqrt(16))                          # → 4.0
print(Counter("banana").most_common(1))       # → [('a', 3)]
print(np.array([1, 2, 3]) * 2)                # → [2 4 6]

4.0
[('a', 3)]
[2 4 6]


**`*args` and `**kwargs`.** You'll mainly need to *read* these in other people's code. `*args` collects any extra positional inputs into a **tuple**, and `**kwargs` collects named inputs into a **dict**:

In [18]:
def total(*args):
    print(args)              # → (1, 2, 3)     a tuple
    return sum(args)

def settings(**kwargs):
    return kwargs            # a dict of name → value

print(total(1, 2, 3))                    # → 6
print(settings(lr=0.01, epochs=5))       # → {'lr': 0.01, 'epochs': 5}

(1, 2, 3)
6
{'lr': 0.01, 'epochs': 5}


**Quick check:** What does `def f(*args): return len(args)` give for `f("a", "b", "c", "d")`?

<details><summary>Answer</summary>

`4`. `args` is the tuple `('a', 'b', 'c', 'd')`.

</details>

---
Next: **`02_numpy_lesson.ipynb`** (Wednesday). The class exercises are numbers 1–4 in **`04_exercises.ipynb`**, for Tuesday.